In [1]:
import os
import ssl
import certifi
import urllib3

# Configure SSL certificates
cert_path = certifi.where()
os.environ['SSL_CERT_FILE'] = cert_path
os.environ['REQUESTS_CA_BUNDLE'] = cert_path
os.environ['AWS_CA_BUNDLE'] = cert_path
os.environ['CURL_CA_BUNDLE'] = cert_path

# Create SSL context with proper certificates
ssl_context = ssl.create_default_context(cafile=cert_path)
ssl._create_default_https_context = lambda: ssl_context

# Disable SSL warnings (optional)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print(f"✅ SSL certificates configured using: {cert_path}")
print("✅ Environment variables set for AWS, requests, and curl")
print("✅ Ready to make secure HTTPS connections!")

✅ SSL certificates configured using: /Users/manojskr/Documents/Code/GitHub/graphrag-toolkit/.venv/lib/python3.10/site-packages/certifi/cacert.pem
✅ Environment variables set for AWS, requests, and curl
✅ Ready to make secure HTTPS connections!


In [2]:
import boto3
import json
import time
import os
from pathlib import Path

# Configuration
APPLICATION_ID = "graphrag-custom"
REGION = "us-east-1"  # Change this to your preferred region
PROVISIONED_MEMORY = "16"

# Initialize AWS clients
neptune = boto3.client("neptune-graph", region_name=REGION)
opensearch = boto3.client("opensearchserverless", region_name=REGION)
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Setting up GraphRAG toolkit in region: {REGION}")
print(f"Application ID: {APPLICATION_ID}")

Setting up GraphRAG toolkit in region: us-east-1
Application ID: graphrag-custom


In [ ]:
# import os
# os.environ['AWS_CA_BUNDLE'] = '/Users/manojskr/Documents/Code/GitHub/graphrag-toolkit/.venv/lib/python3.10/site-packages/certifi/cacert.pem'

In [2]:
# from ssl_helper import create_aws_client, setup_ssl_for_aws
# import json

# # Setup SSL certificates first
# setup_ssl_for_aws()

In [3]:
# import os
# import ssl

# # Use Homebrew certificates path on macOS
# cert_path = '/opt/homebrew/etc/ca-certificates/cert.pem'

# # Check if path exists or try alternate location
# if not os.path.exists(cert_path):
#     cert_path = '/usr/local/etc/ca-certificates/cert.pem'

# # Set environment variables for SSL
# os.environ['SSL_CERT_FILE'] = cert_path
# os.environ['REQUESTS_CA_BUNDLE'] = cert_path
# os.environ['CURL_CA_BUNDLE'] = cert_path

# # Create SSL context with system certificates
# ssl_context = ssl.create_default_context(cafile=cert_path)

# print(f"SSL certificates configured to use: {cert_path}")
# print("Environment variables set:")
# print(f"  SSL_CERT_FILE: {os.environ.get('SSL_CERT_FILE')}")
# print(f"  REQUESTS_CA_BUNDLE: {os.environ.get('REQUESTS_CA_BUNDLE')}")
# print(f"  CURL_CA_BUNDLE: {os.environ.get('CURL_CA_BUNDLE')}")

## Step 1: Create IAM role and policies

In [3]:
print("Creating IAM role and policies...")

try:
    # Initialize IAM client
    iam = boto3.client('iam', region_name=REGION)
    
    # Create IAM role
    role_name = f"{APPLICATION_ID}-role"
    
    assume_role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": [
                        "neptune-graph.amazonaws.com",
                        "aoss.amazonaws.com",
                        "bedrock.amazonaws.com"
                    ]
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }
    
    try:
        role_response = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(assume_role_policy),
            Description=f"Role for GraphRAG toolkit {APPLICATION_ID}"
        )
        print(f"IAM role created: {role_name}")
    except iam.exceptions.EntityAlreadyExistsException:
        print(f"IAM role already exists: {role_name}")
        role_response = iam.get_role(RoleName=role_name)
    
    role_arn = role_response['Role']['Arn']
    
    # Create policy for Neptune Analytics
    neptune_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "neptune-graph:*"
                ],
                "Resource": f"arn:aws:neptune-graph:{REGION}:*:graph/{APPLICATION_ID}-*"
            }
        ]
    }
    
    # Create policy for OpenSearch Serverless
    opensearch_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "aoss:*"
                ],
                "Resource": f"arn:aws:aoss:{REGION}:*:collection/{APPLICATION_ID}-*"
            }
        ]
    }
    
    # Create policy for Bedrock
    bedrock_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel"
                ],
                "Resource": [
                    f"arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-3-haiku-20240307-v1",
                    f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2"
                ]
            }
        ]
    }
    
    # Attach policies to role
    for policy_name, policy_doc in [
        ("neptune", neptune_policy),
        ("opensearch", opensearch_policy),
        ("bedrock", bedrock_policy)
    ]:
        policy_name = f"{APPLICATION_ID}-{policy_name}-policy"
        try:
            iam.put_role_policy(
                RoleName=role_name,
                PolicyName=policy_name,
                PolicyDocument=json.dumps(policy_doc)
            )
            print(f"Attached policy: {policy_name}")
        except Exception as e:
            print(f"Error attaching policy {policy_name}: {e}")
            raise
    
    print(f"\nIAM setup complete!")
    print(f"Role ARN: {role_arn}")
    
except Exception as e:
    print(f"Error in IAM setup: {e}")
    raise


Creating IAM role and policies...
IAM role already exists: graphrag-custom-role
Attached policy: graphrag-custom-neptune-policy
Attached policy: graphrag-custom-opensearch-policy
Attached policy: graphrag-custom-bedrock-policy

IAM setup complete!
Role ARN: arn:aws:iam::533267284022:role/graphrag-custom-role


## Step 2: Create Neptune Analytics Graph

In [4]:
graph_response = neptune.create_graph(
    graphName=f"{APPLICATION_ID}-graph",
    provisionedMemory=int(PROVISIONED_MEMORY),
    deletionProtection=False,
    publicConnectivity=True,
    replicaCount=1,
    vectorSearchConfiguration={
        "dimension": 1024
    }
)

graph_response

{'ResponseMetadata': {'RequestId': 'cae0984d-4737-4e99-8320-ff1e00583ce7',
  'HTTPStatusCode': 201,
  'HTTPHeaders': {'date': 'Mon, 07 Jul 2025 19:25:26 GMT',
   'content-type': 'application/json',
   'content-length': '413',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'cae0984d-4737-4e99-8320-ff1e00583ce7',
   'x-amz-apigw-id': 'NWnOBHNDoAMEHNQ=',
   'x-amzn-trace-id': 'Root=1-686c1f26-00ac97ec1e7b2cac3a79a3f9'},
  'RetryAttempts': 0},
 'id': 'g-25e4k0l6h6',
 'name': 'graphrag-custom-graph',
 'arn': 'arn:aws:neptune-graph:us-east-1:533267284022:graph/g-25e4k0l6h6',
 'status': 'CREATING',
 'createTime': datetime.datetime(2025, 7, 7, 12, 25, 26, 878000, tzinfo=tzlocal()),
 'provisionedMemory': 16,
 'endpoint': 'g-25e4k0l6h6.us-east-1.neptune-graph.amazonaws.com',
 'publicConnectivity': True,
 'vectorSearchConfiguration': {'dimension': 1024},
 'replicaCount': 1,
 'kmsKeyIdentifier': 'AWS_OWNED_KEY',
 'deletionProtection': False}

In [5]:
# Extract ID and endpoint from the response
graph_id = graph_response["id"]  
graph_endpoint = graph_response["endpoint"]

print(f"Neptune Graph created successfully!")
print(f"   Graph ID: {graph_id}")
print(f"   Endpoint: {graph_endpoint}")
print(f"   Status: {graph_response['status']}")

# Wait for graph to be available
print("Waiting for graph to be available...")
waiter = neptune.get_waiter("graph_available")
waiter.wait(graphIdentifier=graph_id)  
print("Graph is now available!")

Neptune Graph created successfully!
   Graph ID: g-25e4k0l6h6
   Endpoint: g-25e4k0l6h6.us-east-1.neptune-graph.amazonaws.com
   Status: CREATING
Waiting for graph to be available...
Graph is now available!


## Step 3: Create OpenSearch Serverless Collection

In [ ]:
# import certifi
# cert_path = certifi.where()

In [6]:

try:
    # Create encryption policy FIRST (required before collection creation)
    encryption_policy = {
        "Rules": [
            {
                "Resource": [f"collection/{APPLICATION_ID}-collection"],
                "ResourceType": "collection"
            }
        ],
        "AWSOwnedKey": True
    }
    
    try:
        opensearch.create_security_policy(
            name=f"{APPLICATION_ID}-encryption",
            policy=json.dumps(encryption_policy),
            type="encryption"
        )
        print("Encryption policy created")
    except opensearch.exceptions.ConflictException:
        print("Encryption policy already exists")
    
    # Create network policy (also required before collection creation)
    network_policy = [
        {
            "Rules": [
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "ResourceType": "dashboard"
                },
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "ResourceType": "collection"
                }
            ],
            "AllowFromPublic": True
        }
    ]
    
    try:
        opensearch.create_security_policy(
            name=f"{APPLICATION_ID}-network",
            policy=json.dumps(network_policy),
            type="network"
        )
        print("Network policy created")
    except opensearch.exceptions.ConflictException:
        print("Network policy already exists")
    
    # Create access policy
    # Get current user ARN
    sts = boto3.client("sts")
    user_arn = sts.get_caller_identity()["Arn"]
    
    access_policy = [
        {
            "Rules": [
                {
                    "Resource": [f"collection/{APPLICATION_ID}-collection"],
                    "Permission": [
                        "aoss:CreateCollectionItems",
                        "aoss:DeleteCollectionItems",
                        "aoss:UpdateCollectionItems",
                        "aoss:DescribeCollectionItems"
                    ],
                    "ResourceType": "collection"
                },
                {
                    "Resource": [f"index/{APPLICATION_ID}-collection/*"],
                    "Permission": [
                        "aoss:CreateIndex",
                        "aoss:DeleteIndex",
                        "aoss:UpdateIndex",
                        "aoss:DescribeIndex",
                        "aoss:ReadDocument",
                        "aoss:WriteDocument"
                    ],
                    "ResourceType": "index"
                }
            ],
            "Principal": [user_arn]
        }
    ]
    
    try:
        opensearch.create_access_policy(
            name=f"{APPLICATION_ID}-access",
            policy=json.dumps(access_policy),
            type="data"
        )
        print("Access policy created")
    except opensearch.exceptions.ConflictException:
        print("Access policy already exists")
    
    print("All security policies are ready!")
    
except Exception as e:
    print(f"Error creating security policies: {e}")
    raise

Encryption policy created
Network policy created
Access policy created
All security policies are ready!


In [7]:

try:
    # Create collection (now that policies exist)
    try:
        collection_response = opensearch.create_collection(
            name=f"{APPLICATION_ID}-collection",
            type="VECTORSEARCH",
            standbyReplicas="DISABLED"
        )
        
        collection_id = collection_response["createCollectionDetail"]["id"]
        
        print(f"OpenSearch Collection created successfully!")
        print(f"   Collection ID: {collection_id}")
        print(f"   Status: {collection_response['createCollectionDetail']['status']}")
        
        # The collection Endpoint is not available immediately after creation
        # We need to wait for the collection to be active and then retrieve it
        
    except opensearch.exceptions.ConflictException:
        print("ℹ️ Collection already exists, retrieving details...")
        # Get existing collection details
        collection_response = opensearch.batch_get_collection(names=[f"{APPLICATION_ID}-collection"])
        collection_detail = collection_response["collectionDetails"][0]
        collection_id = collection_detail["id"]
        
        print(f"Using existing OpenSearch Collection:")
        print(f"   Collection ID: {collection_id}")
        print(f"   Status: {collection_detail['status']}")
    
    # Wait for collection to be active and get the endpoint
    print("Waiting for collection to be active...")
    while True:
        status_response = opensearch.batch_get_collection(names=[f"{APPLICATION_ID}-collection"])
        collection_detail = status_response["collectionDetails"][0]
        status = collection_detail["status"]
        print(f"   Collection status: {status}")
        
        if status == "ACTIVE":
            # Now we can get the endpoint
            collection_endpoint = collection_detail["collectionEndpoint"]
            print(f"   Collection endpoint: {collection_endpoint}")
            break
        elif status in ["FAILED", "DELETED"]:
            raise Exception(f"Collection creation failed with status: {status}")
        
        time.sleep(10)
    
    print("Collection is now active!")
    
except Exception as e:
    print(f"Error creating OpenSearch Collection: {e}")
    raise

OpenSearch Collection created successfully!
   Collection ID: lmzhxvcweuhixd1szx84
   Status: CREATING
Waiting for collection to be active...
   Collection status: CREATING
   Collection status: CREATING
   Collection status: CREATING
   Collection status: ACTIVE
   Collection endpoint: https://lmzhxvcweuhixd1szx84.us-east-1.aoss.amazonaws.com
Collection is now active!


## Step 4: Create .env file

In [8]:
print("Creating .env file for local execution...")

try:
    # Determine region prefix for model names
    region_prefix = REGION.split("-")[0] if "-" in REGION else REGION
    
    # Create .env content with your specified models
    env_content = f"""# AWS Configuration
    AWS_REGION={REGION}

    # Graph Store (Neptune Analytics)
    GRAPH_STORE=neptune-graph://{graph_id}

    # Vector Store (OpenSearch Serverless)
    VECTOR_STORE=aoss://{collection_endpoint}

    # Model Configuration - Using your specified models
    EXTRACTION_MODEL={region_prefix}.anthropic.claude-3-haiku-20240307-v1:0
    EMBEDDINGS_MODEL=amazon.titan-embed-text-v2:0
    EMBEDDINGS_DIMENSIONS=1024
    RESPONSE_MODEL={region_prefix}.anthropic.claude-3-haiku-20240307-v1:0
    EVALUATION_MODEL={region_prefix}.anthropic.claude-3-haiku-20240307-v1:0

    # Neptune Notebook Configuration
    GRAPH_NOTEBOOK_AUTH_MODE=IAM
    GRAPH_NOTEBOOK_SSL=True
    GRAPH_NOTEBOOK_IAM_PROVIDER=ROLE
    GRAPH_NOTEBOOK_PORT=8182
    GRAPH_NOTEBOOK_SERVICE=neptune-graph
    GRAPH_NOTEBOOK_HOST={graph_endpoint}

    # Application ID
    APPLICATION_ID={APPLICATION_ID}
    """
    
    # Write .env file to repository root (go up one level from notebooks directory)
    repo_root = Path.cwd().parent
    env_file_path = repo_root / ".env"
    
    with open(env_file_path, "w") as f:
        f.write(env_content)
    
    print(f"✅ .env file created at: {env_file_path}")
    print("\nEnvironment variables configured:")
    print(f"  - GRAPH_STORE: neptune-graph://{graph_id}")
    print(f"  - VECTOR_STORE: aoss://{collection_endpoint}")
    print(f"  - EXTRACTION_MODEL: {region_prefix}.anthropic.claude-3-haiku-20240307-v1:0")
    print(f"  - EMBEDDINGS_MODEL: amazon.titan-embed-text-v2:0")
    print(f"  - RESPONSE_MODEL: {region_prefix}.anthropic.claude-3-haiku-20240307-v1:0")
    
except Exception as e:
    print(f"❌ Error creating .env file: {e}")
    raise

Creating .env file for local execution...
✅ .env file created at: /Users/manojskr/Documents/Code/GitHub/graphrag-toolkit/.env

Environment variables configured:
  - GRAPH_STORE: neptune-graph://g-25e4k0l6h6
  - VECTOR_STORE: aoss://https://lmzhxvcweuhixd1szx84.us-east-1.aoss.amazonaws.com
  - EXTRACTION_MODEL: us.anthropic.claude-3-haiku-20240307-v1:0
  - EMBEDDINGS_MODEL: amazon.titan-embed-text-v2:0
  - RESPONSE_MODEL: us.anthropic.claude-3-haiku-20240307-v1:0


## Cleanup

In [ ]:

print("Starting cleanup of all resources...")

try:
    # 1. Delete Neptune Analytics Graph
    print("\nCleaning up Neptune Analytics resources...")
    try:
        # Use correct parameters for Neptune Analytics
        neptune.delete_graph(
            graphIdentifier=graph_id,  # Use graphIdentifier, not graphId
            skipSnapshot=True  # Required parameter
        )
        print("Neptune Graph deleted")
    except Exception as e:
        print(f"Error deleting Neptune Graph: {e}")

    # 2. Delete OpenSearch resources
    print("\nCleaning up OpenSearch resources...")
    try:
        # Get collection ID first
        collection_name = f"{APPLICATION_ID}-collection"
        try:
            collection_response = opensearch.batch_get_collection(names=[collection_name])
            collection_id = collection_response["collectionDetails"][0]["id"]
            
            # Delete collection using ID
            opensearch.delete_collection(id=collection_id)  # Use id, not name
            print("OpenSearch Collection deleted")
            
        except Exception as e:
            print(f"Error getting/deleting collection: {e}")
        
        # Delete security policies
        for policy_type in ["network", "encryption"]:
            try:
                opensearch.delete_security_policy(
                    name=f"{APPLICATION_ID}-{policy_type}",
                    type=policy_type
                )
                print(f"{policy_type.title()} policy deleted")
            except Exception as e:
                print(f"Error deleting {policy_type} policy: {e}")
        
        # Delete access policy
        try:
            opensearch.delete_access_policy(
                name=f"{APPLICATION_ID}-access",
                type="data"
            )
            print("Access policy deleted")
        except Exception as e:
            print(f"Error deleting access policy: {e}")
            
    except Exception as e:
        print(f"Error cleaning up OpenSearch resources: {e}")

    # 3. Delete IAM Role and Policies
    print("\nCleaning up IAM resources...")
    try:
        role_name = f"{APPLICATION_ID}-role"
        
        # Delete attached policies
        for policy_name in ["neptune", "opensearch", "bedrock"]:
            try:
                iam.delete_role_policy(
                    RoleName=role_name,
                    PolicyName=f"{APPLICATION_ID}-{policy_name}-policy"
                )
                print(f"Deleted policy: {policy_name}")
            except Exception as e:
                print(f"Error deleting policy {policy_name}: {e}")
        
        # Delete role
        try:
            iam.delete_role(RoleName=role_name)
            print(f"Deleted IAM role: {role_name}")
        except Exception as e:
            print(f"Error deleting IAM role: {e}")
            
    except Exception as e:
        print(f"Error cleaning up IAM resources: {e}")

    # 4. Delete local files (fixed for Jupyter)
    print("\nCleaning up local files...")
    try:
        # Use Path.cwd() instead of __file__ for Jupyter notebooks
        repo_root = Path.cwd().parent
        if repo_root.name == "notebooks":
            repo_root = repo_root.parent
        
        # Delete .env file
        env_file = repo_root / ".env"
        if env_file.exists():
            env_file.unlink()
            print("Deleted .env file")
        
        # Delete extracted directory
        extracted_dir = repo_root / "extracted"
        if extracted_dir.exists():
            import shutil
            shutil.rmtree(extracted_dir)
            print("Deleted extracted directory")
            
        # Delete checkpoint files
        checkpoint_files = ["extraction-checkpoint", "build-checkpoint"]
        for checkpoint in checkpoint_files:
            checkpoint_path = repo_root / checkpoint
            if checkpoint_path.exists():
                shutil.rmtree(checkpoint_path)
                print(f"Deleted {checkpoint}")
            
    except Exception as e:
        print(f"Error cleaning up local files: {e}")

    print("\nCleanup complete!")
    print("Please verify in the AWS Console that all resources have been properly deleted.")
    
except Exception as e:
    print(f"\nError during cleanup: {e}")
    print("Some resources may still exist. Please check the AWS Console to ensure all resources are properly deleted.")
